# Bài 4.3 · Fine-tuning trên ảnh thật

**Câu hỏi của buổi này:** có một model đã học trên ImageNet và vài nghìn ảnh của bài mới, nên học lại bao nhiêu phần của model, và những lỗi nào âm thầm làm hỏng kết quả?

**Oxford-IIIT Pet** có 7 349 ảnh chụp chó mèo thuộc 37 giống, chia sẵn khoảng 3 680 ảnh train và 3 669 ảnh test. Nhiều giống nhìn rất giống nhau.

| Mục | Sẽ thấy |
|---|---|
| 1. Dữ liệu | bảng hai tập, ảnh mẫu |
| 2. Chỉ học head | số tham số được học, accuracy từng epoch |
| 3. Fine-tune cả mạng | accuracy từng epoch, so với chỉ học head |
| 4. Bẫy BatchNorm | quên đặt backbone về `eval()` thì mất bao nhiêu |
| 5. Ảnh bị đoán sai | cặp giống hay nhầm, ảnh kèm nhãn |
| 6. Lưu và nạp lại | logits khớp tuyệt đối |
| 7. Tóm tắt | một bảng các con số chính |

Các dòng ✓ là phép kiểm in ra: chạy lại mà thấy ✗ thì có gì đó sai. Bật **GPU** và **Internet** trong *Session options*; chạy hết khoảng 10 phút. Bài giảng: [ai-canuc80k · bài 4.3](https://ai-canuc80k.ai-web-deploy-20261010.workers.dev/#/lesson/4.3).

In [ ]:
# Ô tiện ích: định dạng bảng, dòng kiểm ✓/✗, nhãn số trên biểu đồ. Không cần đọc để theo bài.
import warnings, contextlib, io, logging
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, HTML

plt.rcParams.update({
    'figure.dpi': 110, 'figure.figsize': (8, 4), 'font.size': 11, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25,
    'legend.frameon': False, 'image.cmap': 'viridis',
})
C = {'blue': '#3b6ea8', 'clay': '#c2603e', 'sand': '#b8892b', 'green': '#4d7337', 'gray': '#8a877f'}
ORDER = [C['blue'], C['clay'], C['sand'], C['green'], C['gray']]
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=ORDER)


def table(df, title=None, fmt=None, best=None, lower_is_better=False, index=False):
    """In một bảng gọn: định dạng số, tô đậm ô tốt nhất của cột `best`."""
    st = df.style.format(fmt, na_rep='–')
    if best is not None:
        col = df[best]
        target = col.min() if lower_is_better else col.max()
        st = st.apply(lambda s: ['font-weight:700;background:#f5e6dd' if (s.name == best and v == target) else '' for v in s], axis=0)
    text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if text_cols:
        st = st.set_properties(subset=text_cols, **{'text-align': 'left'})
    if not index:
        st = st.hide(axis='index')
    st = st.set_table_styles([
        {'selector': 'th', 'props': 'text-align:left;padding:6px 12px;border-bottom:2px solid #d3cebf;font-weight:600'},
        {'selector': 'td', 'props': 'padding:5px 12px;border-bottom:1px solid #e6e2d6'},
        {'selector': 'caption', 'props': 'caption-side:top;text-align:left;font-weight:700;font-size:14px;padding:4px 0 8px;white-space:nowrap'},
    ])
    if title:
        st = st.set_caption(title)
    display(st)


def check(msg, ok):
    """Dòng kiểm tra in ra: ✓ khi đúng, ✗ khi sai (không dừng notebook)."""
    print(('✓ ' if ok else '✗ ') + msg)


def bar_labels(ax, fmt='{:.2f}'):
    """Ghi giá trị lên đầu mỗi thanh của biểu đồ cột."""
    for cont in ax.containers:
        ax.bar_label(cont, labels=[fmt.format(v) for v in cont.datavalues], padding=3, fontsize=9)


@contextlib.contextmanager
def quiet():
    """Ẩn thanh tiến trình và dòng thông báo khi tải dữ liệu hay trọng số."""
    logging.disable(logging.WARNING)
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            yield
    finally:
        logging.disable(logging.NOTSET)


def note(text):
    """Một dòng nhận xét in đậm nhẹ dưới output, tính từ số thật của lần chạy."""
    display(HTML(f'<div style="border-left:3px solid #c2603e;padding:4px 10px;margin:6px 0;color:#444">{text}</div>'))

In [ ]:
import time, random
import torch, torch.nn as nn, torchvision
from torchvision import transforms as T
from torch.utils.data import DataLoader

random.seed(0); np.random.seed(0); torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
EPOCHS, BATCH = 3, 64
table(pd.DataFrame([('torch', torch.__version__), ('torchvision', torchvision.__version__),
                    ('thiết bị', torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU')],
                   columns=['thành phần', 'phiên bản']), 'Môi trường chạy')

## 1. Dữ liệu

Ảnh train được cắt ngẫu nhiên và lật ngang để model thấy nhiều biến thể; ảnh test chỉ thu về 256 rồi cắt giữa 224. Chuẩn hóa lấy đúng mean, std mà ResNet-50 dùng khi học trên ImageNet.

In [ ]:
weights = torchvision.models.ResNet50_Weights.IMAGENET1K_V2
mean, std = weights.transforms().mean, weights.transforms().std
train_tf = T.Compose([T.RandomResizedCrop(224, scale=(0.6, 1.0)), T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize(mean, std)])
test_tf = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), T.Normalize(mean, std)])

root = '/tmp/pets'
with quiet():
    train_ds = torchvision.datasets.OxfordIIITPet(root, split='trainval', transform=train_tf, download=True)
    test_ds = torchvision.datasets.OxfordIIITPet(root, split='test', transform=test_tf, download=True)
view_ds = torchvision.datasets.OxfordIIITPet(root, split='test', transform=T.Compose([T.Resize(256), T.CenterCrop(224)]))
classes = train_ds.classes
train_dl = DataLoader(train_ds, BATCH, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
test_dl = DataLoader(test_ds, BATCH, shuffle=False, num_workers=4, pin_memory=True)
table(pd.DataFrame([('train', len(train_ds)), ('test', len(test_ds)), ('số giống', len(classes))], columns=['', 'số lượng']), 'Oxford-IIIT Pet')
check('37 giống', len(classes) == 37)

idx = random.sample(range(len(view_ds)), 8)
fig, axes = plt.subplots(2, 4, figsize=(13, 6.4))
for ax, i in zip(axes.flat, idx):
    img, y = view_ds[i]
    ax.imshow(img); ax.set_title(classes[y], fontsize=10); ax.axis('off')
fig.suptitle('Ảnh test ngẫu nhiên: nền, ánh sáng, tư thế rất khác nhau', y=1.0)
plt.tight_layout(); plt.show()

In [ ]:
scaler = torch.amp.GradScaler('cuda', enabled=device == 'cuda')

def run_epoch(model, loader, opt=None, sched=None):
    train = opt is not None
    total = correct = 0; loss_sum = 0.0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.set_grad_enabled(train), torch.autocast(device, dtype=torch.float16, enabled=device == 'cuda'):
            logits = model(x)
            loss = nn.functional.cross_entropy(logits, y)
        if train:
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            if sched is not None: sched.step()
        loss_sum += loss.item() * len(y); correct += (logits.argmax(1) == y).sum().item(); total += len(y)
    return loss_sum / total, correct / total

def new_model():
    with quiet():
        m = torchvision.models.resnet50(weights=weights)
    m.fc = nn.Linear(m.fc.in_features, len(classes))
    return m.to(device)

count = lambda m: sum(p.numel() for p in m.parameters() if p.requires_grad)
hist, times = [], {}

## 2. Chỉ học head

Đóng băng toàn bộ backbone, chỉ học lớp `fc` cuối. `requires_grad = False` chặn gradient, nhưng **chưa đủ**: BatchNorm ở chế độ train vẫn cập nhật mean và variance theo từng batch. Vì vậy mỗi epoch đặt cả mạng về `eval()` rồi chỉ bật `train()` cho `fc`.

In [ ]:
head = new_model()
head.requires_grad_(False); head.fc.requires_grad_(True)
opt = torch.optim.AdamW(head.fc.parameters(), lr=1e-3)
n_head, n_all = count(head), sum(p.numel() for p in head.parameters())
t0 = time.time()
for ep in range(EPOCHS):
    head.eval(); head.fc.train()
    tl, ta = run_epoch(head, train_dl, opt)
    head.eval(); vl, va = run_epoch(head, test_dl)
    hist.append(('chỉ học head', ep + 1, ta, va))
times['chỉ học head'] = (time.time() - t0) / EPOCHS
H = pd.DataFrame(hist, columns=['cách', 'epoch', 'train acc', 'test acc'])
table(H, f'Chỉ học head: {n_head:,} trên {n_all:,} tham số ({n_head / n_all:.2%})', fmt={'train acc': '{:.3f}', 'test acc': '{:.3f}'})

## 3. Fine-tune cả mạng

Mở toàn bộ tham số. Backbone đã biết nhìn cạnh, lông, mắt nên chỉ cần chỉnh nhẹ: learning rate 1e-4; head mới học từ đầu nên dùng 1e-3. `OneCycleLR` tăng dần rồi giảm learning rate.

In [ ]:
full = new_model()
groups = [{'params': [p for n, p in full.named_parameters() if not n.startswith('fc.')], 'lr': 1e-4}, {'params': full.fc.parameters(), 'lr': 1e-3}]
opt = torch.optim.AdamW(groups, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[1e-4, 1e-3], total_steps=EPOCHS * len(train_dl))
t0 = time.time()
for ep in range(EPOCHS):
    full.train(); tl, ta = run_epoch(full, train_dl, opt, sched)
    full.eval(); vl, va = run_epoch(full, test_dl)
    hist.append(('cả mạng', ep + 1, ta, va))
times['cả mạng'] = (time.time() - t0) / EPOCHS
H = pd.DataFrame(hist, columns=['cách', 'epoch', 'train acc', 'test acc'])

fig, ax = plt.subplots(figsize=(8, 4))
for (name, g), c in zip(H.groupby('cách', sort=False), [C['sand'], C['blue']]):
    ax.plot(g['epoch'], g['test acc'], marker='o', color=c, lw=2, label=f'{name}: test')
    ax.plot(g['epoch'], g['train acc'], marker='o', color=c, ls='--', lw=1, label=f'{name}: train')
ax.set(xlabel='epoch', ylabel='accuracy', title='Hai cách dùng cùng một ResNet-50'); ax.set_xticks(range(1, EPOCHS + 1)); ax.legend(loc='lower right')
plt.tight_layout(); plt.show()
cmp = pd.DataFrame([(n, cnt, times[n], H[H['cách'] == n]['test acc'].iloc[-1]) for n, cnt in [('chỉ học head', n_head), ('cả mạng', count(full))]],
                   columns=['cách', 'tham số được học', 'giây mỗi epoch', 'test acc'])
table(cmp, f'Sau {EPOCHS} epoch', fmt={'tham số được học': '{:,}', 'giây mỗi epoch': '{:.0f}', 'test acc': '{:.3f}'}, best='test acc')
acc_head, acc_full = cmp['test acc'].iloc[0], cmp['test acc'].iloc[1]
note(f"Chỉ học head đạt <b>{acc_head:.3f}</b> với {n_head:,} tham số; cả mạng đạt <b>{acc_full:.3f}</b> ({acc_full - acc_head:+.3f}) với số tham số gấp {count(full) / n_head:.0f} lần, "
     f"mỗi epoch {times['cả mạng'] / times['chỉ học head']:.1f} lần lâu hơn.")

## 4. Bẫy BatchNorm

Lặp lại mục 2 trong một epoch, nhưng **quên** dòng `head.eval(); head.fc.train()` mà gọi `head.train()` như thường lệ. Trọng số backbone vẫn đóng băng, chỉ có mean, variance chạy của BatchNorm bị cập nhật theo ảnh Pet.

In [ ]:
def head_one_epoch(bn_frozen):
    torch.manual_seed(0)
    m = new_model(); m.requires_grad_(False); m.fc.requires_grad_(True)
    o = torch.optim.AdamW(m.fc.parameters(), lr=1e-3)
    bn0 = m.bn1.running_mean.clone()
    if bn_frozen: m.eval(); m.fc.train()
    else: m.train()
    run_epoch(m, train_dl, o)
    m.eval()
    return run_epoch(m, test_dl)[1], (m.bn1.running_mean - bn0).abs().max().item()

trap = pd.DataFrame([('backbone eval(), chỉ fc train()', *head_one_epoch(True)), ('quên: cả mạng train()', *head_one_epoch(False))],
                    columns=['cách gọi', 'test acc sau 1 epoch', 'mean của BatchNorm đầu tiên đổi'])
table(trap, 'Cùng đóng băng trọng số, chỉ khác chế độ BatchNorm', fmt={'test acc sau 1 epoch': '{:.3f}', 'mean của BatchNorm đầu tiên đổi': '{:.4f}'}, best='test acc sau 1 epoch')
check('khi đặt eval(), thống kê BatchNorm giữ nguyên', trap.iloc[0, 2] == 0)
note(f"Quên đặt <code>eval()</code> làm test acc đổi <b>{trap.iloc[1, 1] - trap.iloc[0, 1]:+.3f}</b>, dù không trọng số nào của backbone được cập nhật: "
     f"chỉ riêng mean, variance chạy của BatchNorm đã trôi theo dữ liệu mới.")

## 5. Ảnh bị đoán sai

Điểm tổng chưa nói model sai ở đâu. Đếm các cặp (giống thật → giống đoán) hay gặp nhất, rồi nhìn tận mắt.

In [ ]:
full.eval(); preds, labels = [], []
with torch.no_grad(), torch.autocast(device, dtype=torch.float16, enabled=device == 'cuda'):
    for x, y in test_dl:
        preds.append(full(x.to(device)).argmax(1).cpu()); labels.append(y)
preds, labels = torch.cat(preds).numpy(), torch.cat(labels).numpy()
wrong = np.flatnonzero(preds != labels)
pairs = pd.Series([(classes[labels[i]], classes[preds[i]]) for i in wrong]).value_counts().head(8)
table(pd.DataFrame([(a, b, n) for (a, b), n in pairs.items()], columns=['giống thật', 'model đoán', 'số ảnh']), f'Sai {len(wrong)} / {len(labels)} ảnh test: tám cặp hay nhầm nhất')

(a, b) = pairs.index[0]
ia = [i for i in wrong if classes[labels[i]] == a and classes[preds[i]] == b][:3]
ref_b = [i for i in range(len(labels)) if labels[i] == classes.index(b) and preds[i] == labels[i]][:3]
fig, axes = plt.subplots(2, 3, figsize=(11, 7.4))
for ax, i in zip(axes[0], ia):
    ax.imshow(view_ds[i][0]); ax.axis('off'); ax.set_title(f'thật: {a}\nđoán: {b}', fontsize=10, color=C['clay'])
for ax, i in zip(axes[1], ref_b):
    ax.imshow(view_ds[i][0]); ax.axis('off'); ax.set_title(f'{b} (đoán đúng)', fontsize=10, color=C['green'])
for ax in axes.flat[len(ia):3]: ax.axis('off')
fig.suptitle(f'Cặp nhầm nhiều nhất: {a} → {b}. Hàng dưới: ảnh {b} thật để so', y=1.0)
plt.tight_layout(); plt.show()
note(f'Model sai <b>{len(wrong)}</b> ảnh ({len(wrong) / len(labels):.1%}); riêng cặp {a} → {b} chiếm {pairs.iloc[0]} ảnh. '
     f'Muốn cải thiện, nhìn vào chính các cặp này trước khi đổi kiến trúc.')

## 6. Lưu rồi nạp lại ở một model mới

Bài nộp chỉ đáng tin khi model nạp lại từ tệp cho ra đúng logits như lúc vừa train.

In [ ]:
torch.save(full.state_dict(), '/tmp/pets_resnet50.pt')
again = new_model(); again.load_state_dict(torch.load('/tmp/pets_resnet50.pt', map_location=device)); again.eval()
x, _ = next(iter(test_dl)); x = x.to(device)
with torch.no_grad():
    diff = (full(x) - again(x)).abs().max().item()
check(f'logits của model nạp lại khớp model vừa train (chênh lớn nhất {diff:.1e})', diff < 1e-4)

## 7. Tóm tắt

In [ ]:
summary = [('chỉ học head: test acc / tham số', f'{acc_head:.3f} / {n_head:,}'),
           ('cả mạng: test acc / tham số', f'{acc_full:.3f} / {count(full):,}'),
           ('quên eval() cho BatchNorm: test acc đổi', f'{trap.iloc[1, 1] - trap.iloc[0, 1]:+.3f}'),
           ('ảnh test sai', f'{len(wrong)} / {len(labels)}'),
           ('cặp nhầm nhiều nhất', f'{a} → {b} ({pairs.iloc[0]} ảnh)'),
           ('chênh logits sau khi nạp lại', f'{diff:.1e}')]
table(pd.DataFrame(summary, columns=['điều đo', 'kết quả']), 'Các con số chính của notebook')

**Thử thêm khi dạy:** tăng `EPOCHS` lên 6; mở thêm `layer4` cùng với head (fine-tune một phần); đổi learning rate backbone từ 1e-4 thành 1e-3 rồi xem accuracy có tụt không.